# Preprocessing Framework for Video Machine Vision under Compression

Trains the neural **preprocessor** and evaluates **preprocessor + CompressAI** against the **CompressAI-only** ablation and **bare H.264 / H.265**, reporting BD-Rate. Two tasks from the paper:

* **Action recognition** — Kinetics-400 (5%), metric = top-1 accuracy.
* **Object tracking** — GOT-10k val, metric = real success-plot AUC (+AO, SR). The default tracker is a self-contained SiamFC; the paper's exact trackers (KYS/DiMP/ATOM/PrDiMP) are optional via pytracking.

**Before running:**
1. Add Input: `rohanmallick/kinetics-train-5per` (action recognition) and/or a GOT-10k **val** dataset (tracking).
2. Settings -> Accelerator = **GPU**.
3. Settings -> Internet = **On** (needed to clone the repo and download pretrained weights).

In [ ]:
# 1) Get the code
!git clone https://github.com/wagur1/preprocessing_upgrade_2.git
%cd preprocessing_upgrade_2

In [ ]:
# 2) Install CompressAI (torch/torchvision/ffmpeg are preinstalled on Kaggle)
!pip install -q compressai
!ffmpeg -version | head -n 1

In [ ]:
# 3) One-shot: build <=3GB index -> train preprocessor -> evaluate vs H.264/H.265
#    Bump --epochs / drop --max-steps for a fuller run.
!python kaggle/run_kaggle.py \
    --config configs/action_recognition.yaml \
    --cap-gb 3 --epochs 3 --max-steps 300 --batch-size 4

In [ ]:
# 4) Show the rate-accuracy curve and BD-Rate summary
import json
from IPython.display import Image, display

display(Image('outputs/eval/rate_accuracy.png'))
res = json.load(open('outputs/eval/results.json'))
print('task:', res['task'], '| metric:', res['metric'])
# the real claim: same-codec preprocessor gain (negative = bit savings)
for pair, v in res['bd_prep_gain'].items():
    print(f"  {pair:26s}: BD-Rate {v['bd_rate_pct']:+.2f}%  BD-Acc {v['bd_accuracy']:+.4f}")

In [ ]:
# 3b) Alternative: paper preset (Zhao et al.) — block-DCT virtual codec + L_D restored.
#     --set overrides any config key; mu=10 + light beta=0.01 matches the paper's alpha=10, alpha*lambda=0.01.
!python kaggle/run_kaggle.py \
    --config configs/action_recognition.yaml \
    --cap-gb 3 --epochs 3 --max-steps 300 --batch-size 4 --seed 0 \
    --set codec.kind=virtual --set loss.mu=10 --set loss.beta=0.01 --set loss.gamma=0.0 \
    --out-dir outputs/paper_s0
!python kaggle/report_ci.py outputs/paper_s0   # same-codec BD-Rate prep gain
# view: json.load('outputs/paper_s0/eval/results.json') as in cell 4, or the image under outputs/paper_s0/eval/

<cell_type>markdown</cell_type>### Object tracking on GOT-10k (paper's 2nd task, full implementation)

Add a GOT-10k **val** dataset as Input (folders with `groundtruth.txt` + numbered frames). This trains the preprocessor with SiamFC's differentiable loss and reports **real success AUC** + BD-Rate vs CompressAI / H.264 / H.265.

In [ ]:
# GOT-10k: build <=3GB index -> train preprocessor (SiamFC loss) -> eval real AUC + BD-Rate
!python kaggle/run_kaggle.py \
    --config configs/tracking.yaml \
    --cap-gb 3 --epochs 3 --max-steps 300 \
    --max-seqs 30 --max-frames 48

# then re-run cell 4 to view the tracking rate-AUC curve + BD-Rate
# (optional) paper's exact trackers: add task.tracker: pytracking:dimp:dimp50 to the config
# after: bash scripts/install_pytracking.sh  (+ network weights)